In [ ]:
import polars as pl
dataset_1 = pl.read_excel('../data/transcribed/object_naming.xlsx')
dataset_2 = pl.read_excel('../data/transcribed/'
                              'object&action_naming.xlsx')


In [ ]:
columns_for_now = [
    'StimSite',
    'Stimulus',
    'RT_start',
    'Response_annot',
    'Response_transcription_annot',
    'Error_type_annot'
]

dataset = pl.concat([
    dataset_1[columns_for_now],
    dataset_2.rename({'RT_start_annot': 'RT_start'})[columns_for_now]
])


In [ ]:
dataset = dataset\
    .with_columns(pl.col("Error_type_annot").str.to_lowercase())


In [ ]:
pl.Config.set_tbl_rows(-1)
print(
*dataset['Error_type_annot']\
    .value_counts()\
    .sort('count', descending=True).rows(),
    sep='\n'
)


In [ ]:
pl.Config.set_tbl_rows(-1)
print(
    dataset['Error_type_annot']\
        .value_counts()\
        .sort('count', descending=True)[1:8]\
        .sum(),
    dataset['Error_type_annot']\
        .value_counts()\
        .sort('count', descending=True)[8:]\
        .sum()
)


In [ ]:
dataset = dataset.with_columns(
    pl.col('Error_type_annot').replace({
    ' ': None,
    's/a': 'speech arrest',
    'фонетическа парафазия': 'фонетическая парафазия',
    'фонетическая парфазия': 'фонетическая парафазия',
    'задрежка': 'задержка',
    'задржка': 'задержка',
    'подбор слова': 'поиск слова'
    })
)


In [ ]:
dataset[['Error_type_annot']] = dataset[['Error_type_annot']].fill_null('нет')

In [ ]:
allowed_types = dataset['Error_type_annot']\
        .value_counts()\
        .sort('count', descending=True)['Error_type_annot'][:8]


In [ ]:
dataset_clean = dataset.filter(
    pl.col('Error_type_annot').is_in(allowed_types)
)


In [ ]:
string_column = dataset_clean[['Error_type_annot']]

dataset_clean_onehot = dataset_clean\
    .to_dummies(columns=['Error_type_annot'])\
    .rename(lambda col: col.replace('Error_type_annot_', ''))

dataset_clean_onehot[['Error_type_annot']] = string_column


In [ ]:
dataset_clean_onehot.head()


In [ ]:
dataset_clean_onehot.write_csv('../data/processed/'
                               'dataset_clean_onehot.csv')
